# CASMI 2026 — CPU retrieval validation

Runs a natural-product source holdout without copying the 2.5M-row training parquet. The visible test file is deliberately not used for model selection.


In [ ]:
from pathlib import Path
import subprocess
import sys

INPUT = Path('/kaggle/input')
DATA_DIR = INPUT / 'competitions/enveda-CASMI26-molecule-id-mass-spectra'
if not (DATA_DIR / 'train.parquet').exists():
    raise FileNotFoundError(f'Competition train.parquet not found under {DATA_DIR}')
source_files = list(INPUT.glob('**/prepare_validation.py'))
if not source_files:
    raise FileNotFoundError('Attach the updated CASMI 2026 Source Code dataset')
SOURCE_ROOT = source_files[0].parent
SRC = SOURCE_ROOT
sys.path.insert(0, str(SRC))
print('Data:', DATA_DIR)
print('Source:', SOURCE_ROOT)
commit_file = SOURCE_ROOT / 'SOURCE_COMMIT.txt'
print('Source version:', commit_file.read_text().strip() if commit_file.exists() else 'missing provenance file')

try:
    import rdkit
    parts = tuple(int(p) for p in rdkit.__version__.split('.')[:3])
except (ImportError, ValueError):
    parts = None
if parts != (2026, 3, 3):
    wheels = [p for p in INPUT.glob('**/*.whl') if 'rdkit' in p.name.lower() or '2026.03.3' in str(p)]
    if not wheels:
        raise RuntimeError('Attach the offline RDKit 2026.03.3 wheel dataset')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--upgrade', str(wheels[0])], check=True)
    import rdkit
    print('RDKit:', rdkit.__version__)


In [ ]:
import subprocess

TRAIN = str(DATA_DIR / 'train.parquet')
WORK = Path('/kaggle/working/validation')
WORK.mkdir(parents=True, exist_ok=True)
source_dir = WORK / 'natural_product_source'
subprocess.run([
    sys.executable, str(SRC / 'prepare_validation.py'), '--train', TRAIN,
    '--output-dir', str(source_dir), '--mode', 'source',
    '--query-source', 'enveda-np-examples',
], check=True)


In [ ]:
# Deferred: structure holdout needs a separate leakage-safe split. Do not run
# the old full-table version while we establish the faster source-holdout baseline.
structure_dir = None
print('Structure holdout deferred; run the natural-product source holdout first.')


In [ ]:
def score_source_holdout(folder):
    predictions = WORK / 'natural_product_source_predictions.csv'
    subprocess.run([
        sys.executable, str(SRC / 'retrieve.py'),
        '--train', TRAIN,
        '--test', str(folder / 'queries.parquet'),
        '--output', str(predictions), '--max-ppm', '30',
        '--fallback-ppm', '250', '--top-k', '25',
        '--exclude-source', 'enveda-np-examples',
    ], check=True)
    subprocess.run([
        sys.executable, str(SRC / 'evaluate_predictions.py'),
        '--queries', str(folder / 'queries.parquet'),
        '--predictions', str(predictions),
    ], check=True)

score_source_holdout(source_dir)


Save the JSON metrics and notebook version with the Git source commit. Keep this as the baseline before testing rankers or GPU models.
